In [1]:
import matplotlib as mpl
import glob
import os
import matplotlib.pyplot as plt
import h5py as h5
import mpl_toolkits as tk
import numpy as np
import netCDF4 as nc
import h5py as h5
from matplotlib.colors import ListedColormap
import pandas as pd
from mpl_toolkits.basemap import Basemap
import metpy.calc
from datetime import date
from datetime import datetime
import stat
from datetime import timedelta
import random
import stat

In [2]:
dpi = 96

def plot_mask_double(namedir, img_array, img_array2, storm_mask, plt_title, 
    my_cmap=None,my_cmap2=None, my_cmap3=None, u_wind=None, v_wind=None, 
    v_max=None, v_min=None,v_max2=None, v_min2=None, line=None, land=True):
    """
    img_array: This is the contour that is being plotted (i.e. TMQ)
    storm_mask: This creates a mask on top of the img_array contour showing the storm labels.  If you do not wish
            to see a predefined mask, you can input np.zeros(img_array.shape) for this field
    plt_title: The title of the plot
    my_cmap: input a custom colormap for the img_array contour.  The default colormap is good though
    u_wind: wind values in the u direction
    v_wind: wind values in the v direction
    """
    # Set alpha
    if my_cmap is None:
        # Choose colormap
        cmap = mpl.cm.viridis
        # Get the colormap colors
        my_cmap = cmap(np.arange(cmap.N))
        alpha = np.linspace(0, 1, cmap.N)
        my_cmap[:,0] = (1-alpha) + alpha * my_cmap[:,0]
        my_cmap[:,1] = (1-alpha) + alpha * my_cmap[:,1]
        my_cmap[:,2] = (1-alpha) + alpha * my_cmap[:,2]

        # Create new colormap
        my_cmap = ListedColormap(my_cmap)

    # l = p['label'] / 100
    p = storm_mask #p['prediction']
    p = np.roll(p,[0,1152//2])
    p1 = (p == 100)
    p2 = (p == 2)

    d = img_array #h['climate']['data'][0,...]
    #d = np.roll(d,[0,1152//2])
    
    d2 = img_array2
    #d2 = np.roll(d2,[0,1152//2])

    lats = np.linspace(-90,90,1152) # changed 768 to 1152
    longs = np.linspace(0, 360,1152)

    def do_fig(figsize):
        fig = plt.figure(figsize=figsize, dpi=dpi)
        ax=fig.add_axes([0,0,1,1])
        ax.axis('off')

        my_map = Basemap(projection='spstere', boundinglat = -20, llcrnrlat=min(lats), lon_0=np.median(longs),
              llcrnrlon=min(longs), urcrnrlat=max(lats), urcrnrlon=max(longs), resolution = 'c', fix_aspect=False)
        xx, yy = np.meshgrid(longs, lats)
        x_map,y_map = my_map(xx,yy)
        my_map.drawcoastlines(color=[0.5,0.5,0.5])

        my_map.contour(x_map,y_map,d,line,cmap=my_cmap, vmax=v_max, vmin=v_min)
        my_map.contourf(x_map,y_map,d2,64,cmap=my_cmap2, vmax=v_max2, vmin=v_min2)
        my_map.drawmeridians(np.arange(0, 360, 60), labels=[0,0,0,1])
        my_map.drawparallels(np.arange(-90, 90, 30), labels =[1,0,0,0])
        if u_wind is not None and v_wind is not None:
            wind_speed = np.sqrt(u_wind**2 + v_wind**2)
            my_map.quiver(x_map[::20,::20],y_map[::20,::20], u_wind[::20,::20], v_wind[::20,::20], wind_speed[::20,::20], alpha=0.5, cmap=my_cmap3)

        if (not land):
            # alpha was at 0.5 - SK change
            my_map.fillcontinents(alpha=0.1) 
        mask_ex = plt.gcf()
        path = namedir + "/" + plt_title
        mask_ex.savefig(path, dpi=dpi, quality=100,pad_inches = 0)
        os.chmod(path, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH)
        plt.clf()

    # do_fig((1152/dpi,768/dpi))
    do_fig((1152/dpi,1152/dpi))

In [3]:
""" This is Sean's plotting function """
def plot_mask_flat(namedir ,img_array, storm_mask, plt_title, my_cmap=None, 
    my_cmap2 = None, u_wind=None, v_wind=None, 
    v_max=None, v_min=None, land=True):
    """
    img_array: This is the contour that is being plotted (i.e. TMQ)
    storm_mask: This creates a mask on top of the img_array contour showing the storm labels.  If you do not wish
            to see a predefined mask, you can input np.zeros(img_array.shape) for this field
    plt_title: The title of the plot
    my_cmap: input a custom colormap for the img_array contour.  The default colormap is good though
    u_wind: wind values in the u direction
    v_wind: wind values in the v direction
    """
    # Set alpha
    if my_cmap is None:
        # Choose colormap
        cmap = mpl.cm.viridis
        # Get the colormap colors
        my_cmap = cmap(np.arange(cmap.N))
        alpha = np.linspace(0, 1, cmap.N)
        my_cmap[:,0] = (1-alpha) + alpha * my_cmap[:,0]
        my_cmap[:,1] = (1-alpha) + alpha * my_cmap[:,1]
        my_cmap[:,2] = (1-alpha) + alpha * my_cmap[:,2]

        # Create new colormap
        my_cmap = ListedColormap(my_cmap)

    # l = p['label'] / 100
    p = storm_mask #p['prediction']
    p = np.roll(p,[0,1152//2])
    p1 = (p == 100)
    p2 = (p == 2)

    d = img_array #h['climate']['data'][0,...]
    #d = np.roll(d,[0,1152//2])

    lats = np.linspace(-90,90,1152) # changed 768 -> 1152
    longs = np.linspace(0, 360,1152)

    def do_fig(figsize):

        fig = plt.figure(figsize=figsize,dpi=dpi)
        ax=fig.add_axes([0,0,1,1])
        ax.axis('off')

        my_map = Basemap(projection='spstere', boundinglat = -20, llcrnrlat=min(lats), lon_0=np.median(longs),
              llcrnrlon=min(longs), urcrnrlat=max(lats), urcrnrlon=max(longs), resolution = 'c', fix_aspect=False)
        xx, yy = np.meshgrid(longs, lats)
        x_map,y_map = my_map(xx,yy)
        my_map.drawcoastlines(color=[0.5,0.5,0.5])

        my_map.contourf(x_map,y_map,d,64,cmap=my_cmap, vmax=v_max, vmin=v_min)
        my_map.drawmeridians(np.arange(0, 360, 60), labels=[0,0,0,1])
        my_map.drawparallels(np.arange(-90, 90, 30), labels =[1,0,0,0])
        if u_wind is not None and v_wind is not None:
            wind_speed = np.sqrt(u_wind**2 + v_wind**2)
            gap = 10
            norm = mpl.colors.Normalize(vmin=0, vmax=20)
            my_map.quiver(x_map[::gap,::gap],y_map[::gap,::gap], u_wind[::gap,::gap], v_wind[::gap,::gap], wind_speed[::gap,::gap],
                alpha=0.9, cmap=my_cmap2, norm=norm, width=0.001, scale=650)
        
        if (not land):
             # alpha was at 0.5 - SK change
            my_map.fillcontinents(alpha=0.1)
        mask_ex = plt.gcf()
        path = namedir + "/" + plt_title
        mask_ex.savefig(path ,dpi=dpi,pad_inches = 0) #quality=100,
        os.chmod(path, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH)
        plt.clf()

    do_fig((1152/dpi,768/dpi))
    # do_fig((1152/dpi,1152/dpi))

In [23]:
""" Select files from folder to convert """

source_channels = ['ivt'] #, 'prw', 'psl', 'u850', 'v850']
sourceName = ['windhusavi'] #, 'prw', 'psl', 'ua850', 'va850']
sourceDir_path = '/glade/scratch/tking/cgnet/high_lat_QC/'
source = dict()
for c in source_channels:
    nc_fps_tmq = glob.glob(sourceDir_path + c + '/polar_' + c + '/*.nc')
    nc_fps_tmq.sort()
    source[c] = nc_fps_tmq

output_channels = ['ivt'] #, 'tmq', 'psl', 'psl_ivt', 'tmq_wind_850']
outputDir_path = '/glade/work/tking/cgnet/ML-extremes/notebooks/PolarARs/'

index = glob.glob('/glade/scratch/tking/06/2001.txt')
index.sort()

In [25]:
cat /glade/scratch/tking/06/2001.txt

56 data-2001-01-02-02-0.jpg
104 data-2001-01-08-02-0.jpg
184 data-2001-01-18-02-0.jpg
192 data-2001-01-19-02-0.jpg
216 data-2001-01-22-02-0.jpg
264 data-2001-01-28-02-0.jpg
288 data-2001-01-31-02-0.jpg
296 data-2001-02-01-02-0.jpg
312 data-2001-02-03-02-0.jpg
336 data-2001-02-06-02-0.jpg
344 data-2001-02-07-02-0.jpg
480 data-2001-02-24-02-0.jpg
592 data-2001-03-10-02-0.jpg
592 data-2001-03-10-02-0.jpg
624 data-2001-03-14-02-0.jpg
624 data-2001-03-14-02-0.jpg
760 data-2001-03-31-02-0.jpg
840 data-2001-04-10-02-0.jpg
848 data-2001-04-11-02-0.jpg
872 data-2001-04-14-02-0.jpg
944 data-2001-04-23-02-0.jpg
968 data-2001-04-26-02-0.jpg
992 data-2001-04-29-02-0.jpg
1008 data-2001-05-01-02-0.jpg
1048 data-2001-05-06-02-0.jpg
1080 data-2001-05-10-02-0.jpg
1088 data-2001-05-11-02-0.jpg
1088 data-2001-05-11-02-0.jpg
1152 data-2001-05-19-02-0.jpg
1192 data-2001-05-24-02-0.jpg
1304 data-2001-06-07-02-0.jpg
1352 data-2001-06-13-02-0.jpg
1368 data-2001-06-15-02-0.jpg
1384 data-2001-06-17-02-0.jpg
1536

In [61]:
for i, idx_txt in enumerate(index): # 2001 - 2003
    print("i, idx_txt: ", i, idx_txt)
    print("The file that this is using is one year before index text file: "+source['ivt'][i])
    print("The file that this should be using in order to correspond with the file name is"+source['ivt'][i+1])
    data_in_ivt = nc.Dataset(source['ivt'][i])
    IVT = data_in_ivt['windhusavi']
print("first time slice in data used (days since 1979): "+str(data_in_ivt['time'][0]))
print("last time slice in data used (days since 1979): "+str(data_in_ivt['time'][-1])) # this spans dec 25 2000-dec 25 2001 (eg, 2001 file, but uses 2000 file)

i, idx_txt:  0 /glade/scratch/tking/06/2001.txt
The file that this is using is one year before index text file: /glade/scratch/tking/cgnet/high_lat_QC/ivt/polar_ivt/windhusavi_3hr_CAM5-1-025degree_All-Hist_est1_v1-0_run002_200001-200012.nc
The file that this should be using in order to correspond with the file name is/glade/scratch/tking/cgnet/high_lat_QC/ivt/polar_ivt/windhusavi_3hr_CAM5-1-025degree_All-Hist_est1_v1-0_run002_200101-200112.nc
first time slice in data used (days since 1979): 7665.0
last time slice in data used (days since 1979): 8029.875


In [ ]:
for i, idx_txt in enumerate(index): # 2001 - 2003
    print(i)
    data_in_ivt = nc.Dataset(source['ivt'][i])
    IVT = data_in_ivt['windhusavi']
    print(IVT)
    # data_in_prw = nc.Dataset(source['prw'][i])
    # TMQ = data_in_prw['prw']
    # data_in_psl = nc.Dataset(source['psl'][i])
    # PSL = data_in_psl['psl']
    # data_in_u850 = nc.Dataset(source['u850'][i])
    # U850 = data_in_u850['ua850']
    # data_in_v850 = nc.Dataset(source['v850'][i])
    # V850 = data_in_v850['va850']

    f = open(idx_txt,"r")
    f1 = f.readlines()

    for x in f1:
        print(x)
        i, fn = x.split()
        """ plot IVT """
        plot_mask_flat(outputDir_path + 'ivt', IVT[int(i)], np.zeros((768, 1152)), fn, 
                       'viridis', v_max=1400,v_min=0, land=False)  # changed from zeros 768, 1152 to 1152, 1152
        # """ plot TMQ """
        # plot_mask_flat(outputDir_path + 'tmq', TMQ[int(i)-1], np.zeros((768, 1152)), fn, 
        #                'viridis', v_max=80,v_min=0, land=False)
        # """ plot PSL """
        # plot_mask_flat(outputDir_path + 'psl', PSL[int(i)-1], np.zeros((768, 1152)), fn, 
        #                'viridis', v_max=101500,v_min=99000, land=False)
        # """ plot IVT contour map + PSL contour line """
        # plot_mask_double(outputDir_path + 'psl_ivt', PSL[int(i)-1], IVT[int(i)],
        #                 np.zeros((768, 1152)), fn, 'cool', 'viridis',
        #                 v_max=101500,v_min=99000,v_max2=1400, v_min2=0,
        #                 land=False, line=9)
        # """ plot TMQ contour map + Wind quiver"""
        # plot_mask_flat(outputDir_path + 'tmq_wind_850', TMQ[int(i)-1], 
        #                 np.zeros((768, 1152)), fn, 'viridis', my_cmap2=mpl.cm.autumn,
        #                 u_wind=U850[int(i)-1][0], v_wind=V850[int(i)-1][0], 
        #                 v_max=80,v_min=0, land=False)

    print("finish one yr", i)

In [68]:
# These plots (in ivt directory) end up looking a bit weird...
# They seem as though they have radial patterns that shouldn't be there

In [123]:
# plt.plot(IVT[0].data[0], IVT[0].data[1])
IVT[0].data[0]

array([398.11334 , 398.139   , 398.03787 , ...,  21.601673,  20.395765,
        19.10002 ], dtype=float32)